In [ ]:
import numpy as np
import pandas as pd

def filter_voltage(df_w0, df_w3, v_min=None, v_max=None):
    """Filters the matched dataframes for a specific voltage range."""
    mask = np.ones(len(df_w0), dtype=bool)
    if v_min is not None:
        mask &= (df_w0['V'] >= v_min)
    if v_max is not None:
        mask &= (df_w0['V'] <= v_max)
    return df_w0[mask], df_w3[mask]

def extract_delta_Q_features(df_w0, df_w3):
    """
    Extracts features based on the difference in discharge capacity: \Delta Q_{w3-w0}(V).
    Features extracted: log(|mean(\Delta Q_{w3-w0}(V))|) and log(|var(\Delta Q_{w3-w0}(V))|)
    """
    delta_Q = df_w3['Q_discharge'].values - df_w0['Q_discharge'].values
    
    mean_delta_Q = np.log(np.abs(np.mean(delta_Q)))
    var_delta_Q = np.log(np.abs(np.var(delta_Q)))
    
    return mean_delta_Q, var_delta_Q

def extract_dQ_dV_features(df_w0, df_w3, v_min=None, v_max=None):
    """
    Extracts features based on the incremental capacity difference: \Delta dQ/dV_{w3-w0}(V).
    Calculates the mean and variance of this difference for a given voltage window.
    """
    df_w0_f, df_w3_f = filter_voltage(df_w0, df_w3, v_min, v_max)
    
    # Calculate dQ/dV using numpy gradient
    V_f = df_w0_f['V'].values
    dQ_dV_w0 = np.gradient(df_w0_f['Q_discharge'].values, V_f)
    dQ_dV_w3 = np.gradient(df_w3_f['Q_discharge'].values, V_f)
    
    delta_dQ_dV = dQ_dV_w3 - dQ_dV_w0
    
    mean_delta_dQ_dV = np.log(np.abs(np.mean(delta_dQ_dV)))
    var_delta_dQ_dV = np.log(np.abs(np.var(delta_dQ_dV)))
    
    return mean_delta_dQ_dV, var_delta_dQ_dV

def extract_initial_capacity_features(df_w0, v_min=None, v_max=None):
    """
    Extracts initial capacity features for week 0: log(Q_{w0}).
    Can be constrained to specific voltage intervals (e.g., 3.6V - 3.9V).
    """
    mask = np.ones(len(df_w0), dtype=bool)
    if v_min is not None:
        mask &= (df_w0['V'] >= v_min)
    if v_max is not None:
        mask &= (df_w0['V'] <= v_max)
        
    Q_w0_filtered = df_w0.loc[mask, 'Q_discharge']
    
    # Capacity is typically the integral or total difference in the interval
    total_capacity = Q_w0_filtered.max() - Q_w0_filtered.min()
    
    return np.log(total_capacity)

def run_all_feature_extractions(df_w0, df_w3):
    """
    Compiles all the requested log-transformed features into a single dictionary.
    """
    features = {}
    
    # Delta Q Features (Full Voltage Range)
    mean_dQ, var_dQ = extract_delta_Q_features(df_w0, df_w3)
    features['log(|mean(Delta_Q(V))|)'] = mean_dQ
    features['log(|var(Delta_Q(V))|)'] = var_dQ
    
    # Incremental Capacity Features (Full Voltage Range)
    mean_dQdV, var_dQdV = extract_dQ_dV_features(df_w0, df_w3)
    features['log(|mean(Delta_dQ/dV(V))|)'] = mean_dQdV
    features['log(|var(Delta_dQ/dV(V))|)'] = var_dQdV
    
    # Incremental Capacity Features (Specific Voltage Intervals)
    intervals = [(3.0, 3.6), (3.6, 3.9), (3.9, 4.2)]
    for v_min, v_max in intervals:
        mean_dQdV_int, var_dQdV_int = extract_dQ_dV_features(df_w0, df_w3, v_min, v_max)
        features[f'log(|mean(Delta_dQ/dV_{v_min}V-{v_max}V)|)'] = mean_dQdV_int
        features[f'log(|var(Delta_dQ/dV_{v_min}V-{v_max}V)|)'] = var_dQdV_int
        
        # Initial Capacity in intervals
        features[f'log(Q_w0_{v_min}V-{v_max}V)'] = extract_initial_capacity_features(df_w0, v_min, v_max)
        
    # Total Initial Capacity
    features['log(Q_w0)'] = extract_initial_capacity_features(df_w0)
    
    return features

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

def calculate_dV_dQ(df, q_col='Q_discharge', v_col='V'):
    """
    Calculates the differential voltage (dV/dQ) directly from the splined V vs Q data.
    """
    V = df[v_col].values
    Q = df[q_col].values
    
    # Direct derivative since the input is already spline smoothed
    dV_dQ = np.gradient(V, Q)
    
    return Q, dV_dQ

def extract_dva_peak_capacities(Q, dV_dQ, prominence=0.05):
    """
    Identifies structural peaks and valleys in the dV/dQ curve.
    """
    # Find peaks (maxima)
    peaks, _ = find_peaks(dV_dQ, prominence=prominence)
    
    # Find valleys (minima) by inverting the curve
    valleys, _ = find_peaks(-dV_dQ, prominence=prominence)
    
    # Sort indices to track features sequentially along the capacity axis
    key_indices = np.sort(np.concatenate((peaks, valleys)))
    
    return Q[key_indices]

def extract_DVA_tracking_features(df_w0, df_w3):
    """
    Extracts the shifts in structural peak capacities between week 3 and week 0.
    """
    Q_w0, dV_dQ_w0 = calculate_dV_dQ(df_w0)
    Q_w3, dV_dQ_w3 = calculate_dV_dQ(df_w3)
    
    # Adjust prominence as needed based on the scale of your splined dV/dQ
    q_markers_w0 = extract_dva_peak_capacities(Q_w0, dV_dQ_w0, prominence=0.1)
    q_markers_w3 = extract_dva_peak_capacities(Q_w3, dV_dQ_w3, prominence=0.1)
    
    features = {}
    
    # Calculate Delta Q for the matched structural markers (\Delta Q^{DVA}_{w3-w0})
    min_markers = min(len(q_markers_w0), len(q_markers_w3))
    
    for i in range(min_markers):
        features[f'Delta_Q_DVA_marker_{i+1}_w3_w0'] = q_markers_w3[i] - q_markers_w0[i]
        
    return features